# Check Claude API usage & cost

- **Part 1** works with your normal API key: tracks tokens and estimated cost for every call you make in this notebook.
- **Part 2** needs an Admin key (`ANTHROPIC_ADMIN_KEY` in `.env`): pulls your organization's real spend from Anthropic's Cost report.

Note: there is no API endpoint for the remaining prepaid credit balance; that is only shown in Console → Settings → Billing.

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv(r"D:\Python_Notebook\Claude_Anthropic\.env", override=True)
print("API key present  :", bool(os.getenv("ANTHROPIC_API_KEY")))
print("Admin key present:", bool(os.getenv("ANTHROPIC_ADMIN_KEY")))

client = anthropic.Anthropic()

## Part 1 – Track cost of your own calls (normal API key)

In [ ]:
# USD per 1M tokens (input, output)
PRICES = {
    "claude-haiku-4-5": (1.00, 5.00),
    "claude-sonnet-5":  (2.00, 10.00),
    "claude-opus-5":    (5.00, 25.00),
}

usage_log = []  # one row per call

def ask(prompt, model="claude-haiku-4-5", max_tokens=1000):
    """Send a prompt, print the answer, and record tokens + estimated cost."""
    msg = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    in_price, out_price = PRICES[model]
    cost = msg.usage.input_tokens / 1e6 * in_price + msg.usage.output_tokens / 1e6 * out_price
    usage_log.append({
        "model": model,
        "input_tokens": msg.usage.input_tokens,
        "output_tokens": msg.usage.output_tokens,
        "cost_usd": cost,
    })
    answer = "".join(b.text for b in msg.content if b.type == "text")
    print(answer)
    print(f"\n[{msg.usage.input_tokens} in / {msg.usage.output_tokens} out tokens  ≈ ${cost:.6f}]")
    return answer

In [ ]:
ask("What is the capital of France?")
ask("Give me 3 tips for learning Python, one line each.");

In [ ]:
import pandas as pd

df = pd.DataFrame(usage_log)
display(df)
print(f"Total tokens: {df.input_tokens.sum() + df.output_tokens.sum()}")
print(f"Total estimated cost this session: ${df.cost_usd.sum():.6f}")

## Part 2 – Organization spend from Anthropic (Admin key required)

Create an Admin key in Console → Settings → Admin keys (needs an organization; not available on individual accounts), then add to `.env`:
```
ANTHROPIC_ADMIN_KEY="sk-ant-admin01-..."
```
Data appears ~5 minutes after requests complete. Amounts come back in **cents**.

In [ ]:
import httpx
from datetime import datetime, timedelta, timezone

DAYS = 7
admin_key = os.getenv("ANTHROPIC_ADMIN_KEY")

if not admin_key:
    print("No ANTHROPIC_ADMIN_KEY in .env - skipping. Check Console → Billing for balance.")
else:
    end = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
    start = end - timedelta(days=DAYS)
    params = {
        "starting_at": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "ending_at": end.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "group_by[]": "description",
    }
    headers = {"x-api-key": admin_key, "anthropic-version": "2023-06-01"}

    rows = []
    while True:
        r = httpx.get("https://api.anthropic.com/v1/organizations/cost_report", params=params, headers=headers)
        r.raise_for_status()
        body = r.json()
        for bucket in body["data"]:
            for row in bucket["results"]:
                rows.append({
                    "date": bucket["starting_at"][:10],
                    "description": row.get("description"),
                    "model": row.get("model"),
                    "usd": float(row["amount"]) / 100,
                })
        if not body.get("has_more"):
            break
        params["page"] = body["next_page"]

    cost_df = pd.DataFrame(rows)
    if cost_df.empty:
        print(f"No spend in the last {DAYS} days.")
    else:
        display(cost_df.groupby("date")["usd"].sum().round(4).to_frame("usd_per_day"))
        print(f"Total spend last {DAYS} days: ${cost_df.usd.sum():.4f}")